# Data Validation — Full Pipeline Health Check

Runs **after** all phase notebooks have completed.
Validates every DuckDB table and every CSV export against expected contracts.

| Symbol | Meaning |
|---|---|
| ✅ PASS | Within expected bounds |
| ⚠️ WARN | Unexpected but not blocking |
| ❌ FAIL | Contract violated — pipeline output is unreliable |

Run this any time you re-execute the pipeline to confirm nothing silently broke.

In [ ]:
import duckdb
import polars as pl
import os
from dataclasses import dataclass
from typing import Literal

DB_PATH  = 'data/apex.duckdb'
DATA_DIR = 'data'

@dataclass
class Result:
    phase : str
    table : str
    check : str
    status: str
    detail: str

results: list = []

def check(phase, table, name, passed, detail, warn_only=False):
    status = 'PASS' if passed else ('WARN' if warn_only else 'FAIL')
    results.append(Result(phase, table, name, status, detail))

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f'Database not found: {DB_PATH}\nRun all phase notebooks first (Phase 1 -> Phase 2 -> Phase 3).'
    )

conn = duckdb.connect(DB_PATH, read_only=True)
TABLES = {t[0] for t in conn.execute('SHOW TABLES').fetchall()}
print('Connected to DuckDB.')
print(f'Tables found: {sorted(TABLES)}')

---
## Phase 1 — raw_logs

In [ ]:
P, T = 'Phase 1', 'raw_logs'
check(P, T, 'Table exists', T in TABLES, f'Found: {T in TABLES}')

if T in TABLES:
    n = conn.execute(f'SELECT COUNT(*) FROM {T}').fetchone()[0]
    check(P, T, 'Row count = 15,000', n == 15_000, f'Got {n:,}')

    nulls = conn.execute(f'''
        SELECT SUM(CASE WHEN timestamp IS NULL THEN 1 ELSE 0 END)
             + SUM(CASE WHEN query_type IS NULL THEN 1 ELSE 0 END)
             + SUM(CASE WHEN target_table IS NULL THEN 1 ELSE 0 END)
             + SUM(CASE WHEN execution_time_ms IS NULL THEN 1 ELSE 0 END)
             + SUM(CASE WHEN server_cpu_load IS NULL THEN 1 ELSE 0 END)
             + SUM(CASE WHEN status IS NULL THEN 1 ELSE 0 END)
        FROM {T}
    ''').fetchone()[0]
    check(P, T, 'No nulls in any column', nulls == 0, f'{nulls} nulls found')

    bad_exec = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE execution_time_ms <= 0').fetchone()[0]
    check(P, T, 'execution_time_ms > 0', bad_exec == 0, f'{bad_exec} invalid rows')

    bad_cpu = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE server_cpu_load < 0 OR server_cpu_load > 100').fetchone()[0]
    check(P, T, 'server_cpu_load in [0, 100]', bad_cpu == 0, f'{bad_cpu} out of range')

    statuses = {r[0] for r in conn.execute(f'SELECT DISTINCT status FROM {T}').fetchall()}
    check(P, T, 'status values valid', statuses <= {'SUCCESS', 'TIMEOUT'}, f'Found: {statuses}')

    qtypes = {r[0] for r in conn.execute(f'SELECT DISTINCT query_type FROM {T}').fetchall()}
    check(P, T, 'query_type values valid', qtypes <= {'SELECT','INSERT','UPDATE','DELETE'}, f'Found: {qtypes}')

    tbls = {r[0] for r in conn.execute(f'SELECT DISTINCT target_table FROM {T}').fetchall()}
    allowed = {'orders','customers','products','order_items','order_reviews'}
    check(P, T, 'target_table values valid', tbls <= allowed, f'Found: {tbls}')

    timeout_rate = conn.execute(f"SELECT AVG(CASE WHEN status='TIMEOUT' THEN 1.0 ELSE 0 END) FROM {T}").fetchone()[0]
    check(P, T, 'Timeout rate in [1%, 10%]', 0.01 <= timeout_rate <= 0.10, f'{timeout_rate:.2%}', warn_only=True)

print('Phase 1 / raw_logs done.')

---
## Phase 1 — clean_logs

In [ ]:
P, T = 'Phase 1', 'clean_logs'
check(P, T, 'Table exists', T in TABLES, f'Found: {T in TABLES}')

if T in TABLES:
    n     = conn.execute(f'SELECT COUNT(*) FROM {T}').fetchone()[0]
    raw_n = conn.execute('SELECT COUNT(*) FROM raw_logs').fetchone()[0] if 'raw_logs' in TABLES else -1
    check(P, T, 'Row count matches raw_logs', n == raw_n, f'clean={n:,}  raw={raw_n:,}')

    risk_vals = {r[0] for r in conn.execute(f'SELECT DISTINCT risk_level FROM {T}').fetchall()}
    check(P, T, 'risk_level values valid', risk_vals <= {'CRITICAL','WARNING','OK'}, f'Found: {risk_vals}')

    bad = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE is_anomaly NOT IN (0, 1)').fetchone()[0]
    check(P, T, 'is_anomaly is binary', bad == 0, f'{bad} invalid values')

    null_risk = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE risk_level IS NULL OR is_anomaly IS NULL OR anomaly_score IS NULL').fetchone()[0]
    check(P, T, 'No nulls in risk columns', null_risk == 0, f'{null_risk} nulls')

    crit = conn.execute(f"SELECT AVG(CASE WHEN risk_level='CRITICAL' THEN 1.0 ELSE 0 END) FROM {T}").fetchone()[0]
    check(P, T, 'CRITICAL share < 15%', crit < 0.15, f'{crit:.2%}', warn_only=True)

print('Phase 1 / clean_logs done.')

---
## Phase 2 — delivery

In [ ]:
P, T = 'Phase 2', 'delivery'
check(P, T, 'Table exists', T in TABLES, f'Found: {T in TABLES}')

if T in TABLES:
    n = conn.execute(f'SELECT COUNT(*) FROM {T}').fetchone()[0]
    check(P, T, 'Row count > 90,000', n > 90_000, f'Got {n:,}')

    nulls = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE customer_state IS NULL OR delivery_days IS NULL OR is_late IS NULL').fetchone()[0]
    check(P, T, 'No nulls in key columns', nulls == 0, f'{nulls} nulls')

    neg = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE delivery_days < 0').fetchone()[0]
    check(P, T, 'delivery_days >= 0', neg == 0, f'{neg} negative values')

    extreme = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE delivery_days > 365').fetchone()[0]
    check(P, T, 'delivery_days < 365 days', extreme == 0, f'{extreme} extreme rows', warn_only=True)

    bad_late = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE is_late NOT IN (0, 1)').fetchone()[0]
    check(P, T, 'is_late is binary', bad_late == 0, f'{bad_late} invalid values')

    late_rate = conn.execute(f'SELECT AVG(CAST(is_late AS DOUBLE)) FROM {T}').fetchone()[0]
    check(P, T, 'Late rate in [2%, 20%]', 0.02 <= late_rate <= 0.20, f'{late_rate:.2%}', warn_only=True)

    avg_days = conn.execute(f'SELECT AVG(delivery_days) FROM {T}').fetchone()[0]
    check(P, T, 'Avg delivery in [5, 30] days', 5 <= avg_days <= 30, f'{avg_days:.1f} days', warn_only=True)

    # New: check state_name and country columns exist (added for Power BI geocoding)
    cols = {r[0] for r in conn.execute(f'PRAGMA table_info({T})').fetchall()}
    check(P, T, 'state_name column present', 'state_name' in cols, f'Cols: {cols}')
    check(P, T, 'country column present',    'country'    in cols, f'Cols: {cols}')

    if 'country' in cols:
        countries = {r[0] for r in conn.execute(f'SELECT DISTINCT country FROM {T}').fetchall()}
        check(P, T, 'country = Brazil only', countries == {'Brazil'}, f'Found: {countries}')

    # Duplicate order_id check
    total_rows  = conn.execute(f'SELECT COUNT(*) FROM {T}').fetchone()[0]
    unique_oids = conn.execute(f'SELECT COUNT(DISTINCT order_id) FROM {T}').fetchone()[0]
    check(P, T, 'No duplicate order_ids', total_rows == unique_oids,
          f'total={total_rows:,}  unique={unique_oids:,}', warn_only=True)

print('Phase 2 / delivery done.')


---
## Phase 3 — sentiment

In [ ]:
P, T = 'Phase 3', 'sentiment'
check(P, T, 'Table exists', T in TABLES, f'Found: {T in TABLES}')

if T in TABLES:
    n = conn.execute(f'SELECT COUNT(*) FROM {T}').fetchone()[0]
    check(P, T, 'Row count > 0', n > 0, f'Got {n:,}')

    sent_vals = {r[0] for r in conn.execute(f'SELECT DISTINCT sentiment FROM {T}').fetchall()}
    check(P, T, 'sentiment values valid', sent_vals <= {'Positive','Neutral','Negative'}, f'Found: {sent_vals}')

    bad_score = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE review_score < 1 OR review_score > 5').fetchone()[0]
    check(P, T, 'review_score in [1, 5]', bad_score == 0, f'{bad_score} out-of-range')

    null_sent = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE sentiment IS NULL OR review_score IS NULL').fetchone()[0]
    check(P, T, 'No nulls in sentiment/review_score', null_sent == 0, f'{null_sent} nulls')

    pos_5 = conn.execute(f"SELECT AVG(CASE WHEN sentiment IN ('Positive','Neutral') THEN 1.0 ELSE 0 END) FROM {T} WHERE review_score = 5").fetchone()[0] or 0
    check(P, T, '5-star mostly Positive/Neutral (>60%)', pos_5 >= 0.60, f'{pos_5:.1%}', warn_only=True)

    neg_1 = conn.execute(f"SELECT AVG(CASE WHEN sentiment IN ('Negative','Neutral') THEN 1.0 ELSE 0 END) FROM {T} WHERE review_score = 1").fetchone()[0] or 0
    check(P, T, '1-star mostly Negative/Neutral (>60%)', neg_1 >= 0.60, f'{neg_1:.1%}', warn_only=True)

print('Phase 3 / sentiment done.')

---
## Phase 3 — category_sentiment

In [ ]:
P, T = 'Phase 3', 'category_sentiment'
check(P, T, 'Table exists', T in TABLES, f'Found: {T in TABLES}')

if T in TABLES:
    n = conn.execute(f'SELECT COUNT(*) FROM {T}').fetchone()[0]
    check(P, T, 'Row count > 0', n > 0, f'Got {n:,}')

    nulls = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE category IS NULL OR sentiment IS NULL').fetchone()[0]
    check(P, T, 'No nulls in category/sentiment', nulls == 0, f'{nulls} nulls')

    vals = {r[0] for r in conn.execute(f'SELECT DISTINCT sentiment FROM {T}').fetchall()}
    check(P, T, 'sentiment values valid', vals <= {'Positive','Neutral','Negative'}, f'Found: {vals}')

    bad_stars = conn.execute(f'SELECT COUNT(*) FROM {T} WHERE star_rating < 1 OR star_rating > 5').fetchone()[0]
    check(P, T, 'star_rating in [1, 5]', bad_stars == 0, f'{bad_stars} out-of-range')

    cat_count = conn.execute(f'SELECT COUNT(DISTINCT category) FROM {T}').fetchone()[0]
    check(P, T, 'At least 5 distinct categories', cat_count >= 5, f'Found {cat_count}')

print('Phase 3 / category_sentiment done.')

---
## CSV Exports

In [ ]:
P = 'Exports'
EXPECTED = {
    'synthetic_sql_logs.csv'          : {'min_rows': 14_000, 'cols': ['timestamp','execution_time_ms','server_cpu_load','status']},
    'processed_logs.csv'              : {'min_rows': 14_000, 'cols': ['risk_level','is_anomaly','anomaly_score']},
    'delivery_analysis.csv'           : {'min_rows': 90_000, 'cols': ['customer_state','state_name','country','delivery_days','is_late']},
    'enriched_customer_sentiment.csv' : {'min_rows': 100,    'cols': ['review_score','sentiment']},
    'category_sentiment.csv'          : {'min_rows': 50,     'cols': ['category','sentiment','star_rating']},
    'category_stats.csv'              : {'min_rows': 3,      'cols': ['category','avg_stars','pct_positive','pct_negative','pct_neutral','review_count']},
}

for fname, contract in EXPECTED.items():
    path = os.path.join(DATA_DIR, fname)
    exists = os.path.exists(path)
    check(P, fname, 'File exists', exists, path)
    if exists:
        df_csv = pl.read_csv(path, infer_schema_length=100)
        check(P, fname, f'Row count >= {contract["min_rows"]:,}', len(df_csv) >= contract['min_rows'], f'Got {len(df_csv):,}')
        missing = [c for c in contract['cols'] if c not in df_csv.columns]
        check(P, fname, 'Required columns present', not missing, f'Missing: {missing}' if missing else 'All present')

        # Extra check: no completely empty columns
        empty_cols = [c for c in df_csv.columns if df_csv[c].null_count() == len(df_csv)]
        check(P, fname, 'No fully-null columns', not empty_cols, f'Empty cols: {empty_cols}' if empty_cols else 'None', warn_only=True)

print('CSV checks done.')


---
## Cross-Phase Integrity

In [ ]:
P = 'Cross-phase'

if 'raw_logs' in TABLES and 'clean_logs' in TABLES:
    rn = conn.execute('SELECT COUNT(*) FROM raw_logs').fetchone()[0]
    cn = conn.execute('SELECT COUNT(*) FROM clean_logs').fetchone()[0]
    check(P, 'raw_logs <-> clean_logs', 'Row counts match', rn == cn, f'raw={rn:,}  clean={cn:,}')

if 'clean_logs' in TABLES:
    unclass = conn.execute('SELECT COUNT(*) FROM clean_logs WHERE risk_level IS NULL').fetchone()[0]
    check(P, 'clean_logs', 'All rows have a risk_level', unclass == 0, f'{unclass} unclassified')

if 'sentiment' in TABLES:
    total  = conn.execute('SELECT COUNT(*) FROM sentiment').fetchone()[0]
    unique = conn.execute('SELECT COUNT(DISTINCT review_id) FROM sentiment').fetchone()[0]
    check(P, 'sentiment', 'No duplicate review_ids', total == unique, f'total={total:,}  unique={unique:,}')

conn.close()
print('Cross-phase checks done.')

---
## Validation Report

In [ ]:
ICONS = {'PASS': 'PASS', 'WARN': 'WARN', 'FAIL': 'FAIL'}

passes = [r for r in results if r.status == 'PASS']
warns  = [r for r in results if r.status == 'WARN']
fails  = [r for r in results if r.status == 'FAIL']

print(f'{"Phase":<14} {"Table":<32} {"Check":<44} {"Status":<6} Detail')
print('-' * 120)
for r in results:
    icon = {'PASS': '✅', 'WARN': '⚠️ ', 'FAIL': '❌'}[r.status]
    print(f'{r.phase:<14} {r.table:<32} {r.check:<44} {icon} {r.detail}')

print()
print('=' * 55)
print('  VALIDATION SUMMARY')
print('=' * 55)
print(f'  Total checks : {len(results)}')
print(f'  PASS         : {len(passes)}')
print(f'  WARN         : {len(warns)}')
print(f'  FAIL         : {len(fails)}')
print('=' * 55)

if fails:
    print()
    print('FAILURES — fix before trusting pipeline output:')
    for r in fails:
        print(f'  FAIL  [{r.phase} / {r.table}]  {r.check}  ->  {r.detail}')

if warns:
    print()
    print('WARNINGS — investigate if results look unexpected:')
    for r in warns:
        print(f'  WARN  [{r.phase} / {r.table}]  {r.check}  ->  {r.detail}')

if not fails:
    print()
    print('  All checks passed. Pipeline output is valid.')
    print('  Safe to refresh Power BI.')